# Part 4：Nsight Trace 與 HPCToolkit Proxy 消融

## 研究問題

加入 CUDA trace summaries 或 HPCToolkit artifact proxies，是否穩定優於 compact GPU metrics？

## 目前結論

沒有穩定超過 Part 3 的 3.70% MAPE。這不能證明 HPCToolkit 無效，只表示現有 proxy 粒度與樣本量不足。

> 本 Notebook 新增於既有研究產物之外，不會取代原始 Python scripts 或 CSV／Markdown 報告。

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display


def find_project_root() -> Path:
    """Find the repository whether Jupyter starts at the repo or notebook folder."""
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "analysis").is_dir() and (candidate / "analysis_outputs").is_dir():
            return candidate
    raise FileNotFoundError("找不到專案根目錄；請從本 repository 內啟動 Jupyter。")


PROJECT_ROOT = find_project_root()
OUTPUT_ROOT = PROJECT_ROOT / "analysis_outputs"
RUN_EXPENSIVE = False  # 明確改成 True 才會重跑分析腳本。


def project_path(relative: str) -> Path:
    return PROJECT_ROOT / relative


def load_csv(relative: str, **kwargs) -> pd.DataFrame:
    path = project_path(relative)
    if not path.exists():
        raise FileNotFoundError(f"缺少資料：{path}")
    return pd.read_csv(path, **kwargs)


def show_markdown(relative: str) -> None:
    path = project_path(relative)
    display(Markdown(path.read_text(encoding="utf-8")))


def inspect_inputs(paths) -> pd.DataFrame:
    rows = []
    for relative in paths:
        path = project_path(relative)
        rows.append({
            "path": relative,
            "exists": path.exists(),
            "type": "directory" if path.is_dir() else "file",
            "size_mib": round(path.stat().st_size / 1024**2, 3) if path.is_file() else None,
        })
    return pd.DataFrame(rows)


def run_commands(commands) -> None:
    """Show commands by default; run only after explicit opt-in."""
    for args in commands:
        command = [sys.executable, *args]
        print(" ".join(map(str, command)))
        if RUN_EXPENSIVE:
            subprocess.run(command, cwd=PROJECT_ROOT, check=True)


print(f"PROJECT_ROOT = {PROJECT_ROOT}")
print(f"RUN_EXPENSIVE = {RUN_EXPENSIVE}")

## 實驗步驟

1. 彙整 CUDA kernel、API 與 memcpy trace。
2. 盤點 HPCToolkit postprocess artifacts。
3. 建立 trace／proxy feature families。
4. 與 Part 3 compact GPU baseline 做同 fold 消融。
5. 比較增量訊號與 variance。

## 輸入與 provenance

下列 cell 只檢查輸入是否存在，不修改資料。

In [ ]:
INPUTS = [
    "analysis_outputs/nsys_trace_features.csv",
    "analysis_outputs/hpctoolkit_postprocess_inventory.csv",
    "analysis_outputs/nsys_gpu_metrics_features.csv",
]
display(inspect_inputs(INPUTS))

## 重現入口

下列命令依現有 pipeline 排列。預設只列印；將 RUN_EXPENSIVE 明確改成 True 才會執行。

In [ ]:
COMMANDS = [
    ["analysis/extract_nsys_trace.py"],
    ["analysis/inspect_hpctoolkit_postprocess.py"],
    ["analysis/run_trace_hpct_models.py"],
]
run_commands(COMMANDS)

## 結果表

In [ ]:
METRICS_PATH = "analysis_outputs/model_experiments/part4_trace_hpct_models/metrics.csv"
PREDICTIONS_PATH = "analysis_outputs/model_experiments/part4_trace_hpct_models/predictions.csv"

metrics = load_csv(METRICS_PATH)
sort_column = "mape_percent" if "mape_percent" in metrics.columns else "mean_period_ape_percent"
display(metrics.sort_values(sort_column).reset_index(drop=True))

if "direction" in metrics.columns and "model" in metrics.columns:
    plot_data = metrics.sort_values(sort_column).groupby("direction", as_index=False).first()
    ax = plot_data.plot.bar(x="direction", y=sort_column, legend=False, figsize=(8, 4))
    ax.set_ylabel(sort_column)
    ax.set_title("各方向最佳列（依表內指標排序）")
    plt.xticks(rotation=20, ha="right")
    plt.tight_layout()
    plt.show()

## OOF 預測與 worst cases

顯示現有 prediction table 中 APE 最大的列，並提供 actual-vs-predicted 圖。

In [ ]:
METRICS_PATH = "analysis_outputs/model_experiments/part4_trace_hpct_models/metrics.csv"
PREDICTIONS_PATH = "analysis_outputs/model_experiments/part4_trace_hpct_models/predictions.csv"

predictions = load_csv(PREDICTIONS_PATH)
display(predictions.sort_values("ape_percent", ascending=False).head(15).reset_index(drop=True))

required = {"actual_ms", "predicted_ms", "ape_percent"}
if required.issubset(predictions.columns):
    view = predictions.copy()
    if "deployment_valid" in view.columns:
        view = view[view["deployment_valid"].astype(str).str.lower().eq("true")]
    ax = view.plot.scatter(x="actual_ms", y="predicted_ms", c="ape_percent", cmap="viridis", figsize=(6, 5))
    low = min(view["actual_ms"].min(), view["predicted_ms"].min())
    high = max(view["actual_ms"].max(), view["predicted_ms"].max())
    ax.plot([low, high], [low, high], "--", color="black", linewidth=1)
    ax.set_title("OOF prediction：actual vs predicted")
    plt.tight_layout()
    plt.show()

## 既有最終報告快照

> 來源：`analysis_outputs/model_experiments/part4_trace_hpct_models/report.md`。這是建立 Notebook 時嵌入的結果快照；重新執行分析後，請以磁碟上的最新報告為準。

# Part 4: Nsight trace and HPCToolkit proxy ablations

All model inputs are observable on the source device during profiling. The target-device unprofiled baseline runtime is label-only, and `source_ground_truth_iter_ms` is excluded.

Nsight trace inputs contain kernel, idle-gap, memory-operation, synchronization, and CUDA API counts/times. Current HPCToolkit inputs are artifact-structure proxies (database sizes, structure/correlation counts, and accelerator trace records), not extracted numeric CCT performance metrics.

Outer evaluation is leave-one-workload-out. Scaling, variance filtering, and Ridge alpha selection occur only inside each outer training fold.

| Direction | Model | Inputs | MAE ms | RMSE ms | MAPE | Median APE | Max APE | R2 |
|---|---|---:|---:|---:|---:|---:|---:|---:|
| RTX4090_to_RTX5090 | static_compact_gpu_ridge | 15 | 1.284 | 1.549 | 3.14% | 2.17% | 10.74% | 0.9935 |
| RTX4090_to_RTX5090 | static_trace_ridge | 19 | 1.353 | 1.711 | 3.28% | 2.84% | 9.19% | 0.9921 |
| RTX4090_to_RTX5090 | static_compact_gpu_hpct_proxy_ridge | 21 | 1.436 | 2.026 | 3.33% | 2.78% | 10.68% | 0.9889 |
| RTX4090_to_RTX5090 | static_trace_hpct_proxy_ridge | 25 | 1.422 | 1.800 | 3.48% | 3.10% | 9.64% | 0.9912 |
| RTX4090_to_RTX5090 | trace_only_ridge | 16 | 1.608 | 1.986 | 3.89% | 3.70% | 11.00% | 0.9893 |
| RTX4090_to_RTX5090 | static_compact_gpu_trace_ridge | 31 | 2.115 | 3.650 | 4.32% | 2.87% | 15.08% | 0.9640 |
| RTX4090_to_RTX5090 | full_profile_fusion_ridge | 37 | 2.183 | 3.509 | 4.66% | 3.38% | 14.11% | 0.9667 |
| RTX4090_to_RTX5090 | static_only_ridge | 3 | 6.966 | 10.439 | 14.45% | 13.51% | 38.35% | 0.7055 |
| RTX4090_to_RTX5090 | static_hpct_proxy_ridge | 9 | 7.169 | 10.152 | 15.66% | 14.31% | 36.22% | 0.7215 |
| RTX5090_to_RTX4090 | static_compact_gpu_ridge | 15 | 2.028 | 2.429 | 4.26% | 3.25% | 10.22% | 0.9900 |
| RTX5090_to_RTX4090 | static_compact_gpu_hpct_proxy_ridge | 21 | 2.196 | 2.776 | 4.60% | 3.61% | 11.53% | 0.9869 |
| RTX5090_to_RTX4090 | static_compact_gpu_trace_ridge | 31 | 2.068 | 2.913 | 4.61% | 3.22% | 16.09% | 0.9856 |
| RTX5090_to_RTX4090 | static_trace_ridge | 19 | 2.490 | 4.128 | 4.83% | 3.73% | 21.41% | 0.9711 |
| RTX5090_to_RTX4090 | full_profile_fusion_ridge | 37 | 2.231 | 3.036 | 4.87% | 3.49% | 16.44% | 0.9843 |
| RTX5090_to_RTX4090 | trace_only_ridge | 16 | 2.719 | 4.426 | 4.99% | 4.22% | 20.72% | 0.9667 |
| RTX5090_to_RTX4090 | static_trace_hpct_proxy_ridge | 25 | 2.314 | 3.293 | 5.01% | 3.54% | 21.77% | 0.9816 |
| RTX5090_to_RTX4090 | static_only_ridge | 3 | 10.976 | 16.141 | 19.72% | 18.53% | 44.20% | 0.5574 |
| RTX5090_to_RTX4090 | static_hpct_proxy_ridge | 9 | 11.441 | 17.850 | 20.29% | 19.58% | 52.29% | 0.4587 |
| both_directions_combined | static_compact_gpu_ridge | 15 | 1.656 | 2.037 | 3.70% | 2.58% | 10.74% | 0.9915 |
| both_directions_combined | static_compact_gpu_hpct_proxy_ridge | 21 | 1.816 | 2.430 | 3.96% | 3.30% | 11.53% | 0.9879 |
| both_directions_combined | static_trace_ridge | 19 | 1.921 | 3.159 | 4.05% | 3.16% | 21.41% | 0.9795 |
| both_directions_combined | static_trace_hpct_proxy_ridge | 25 | 1.868 | 2.654 | 4.24% | 3.20% | 21.77% | 0.9856 |
| both_directions_combined | trace_only_ridge | 16 | 2.164 | 3.430 | 4.44% | 3.77% | 20.72% | 0.9759 |
| both_directions_combined | static_compact_gpu_trace_ridge | 31 | 2.091 | 3.302 | 4.46% | 3.11% | 16.09% | 0.9777 |
| both_directions_combined | full_profile_fusion_ridge | 37 | 2.207 | 3.281 | 4.76% | 3.38% | 16.44% | 0.9779 |
| both_directions_combined | static_only_ridge | 3 | 8.971 | 13.592 | 17.08% | 16.15% | 44.20% | 0.6213 |
| both_directions_combined | static_hpct_proxy_ridge | 9 | 9.305 | 14.521 | 17.97% | 15.70% | 52.29% | 0.5678 |

## Comparison with Part 3

| Direction | Part 3 best | Part 3 MAPE | Part 4 best | Part 4 MAPE | Relative change |
|---|---|---:|---|---:|---:|
| RTX4090_to_RTX5090 | compact_gpu_ridge | 3.14% | static_compact_gpu_ridge | 3.14% | +0.0% |
| RTX5090_to_RTX4090 | compact_gpu_ridge | 4.26% | static_compact_gpu_ridge | 4.26% | +0.0% |

## Hyperparameter selection frequency

- RTX4090_to_RTX5090 / full_profile_fusion_ridge: `{"model__alpha": 10.0}`: 10; `{"model__alpha": 0.1}`: 7; `{"model__alpha": 1.0}`: 6; `{"model__alpha": 0.01}`: 1
- RTX4090_to_RTX5090 / static_compact_gpu_hpct_proxy_ridge: `{"model__alpha": 0.1}`: 17; `{"model__alpha": 1.0}`: 7
- RTX4090_to_RTX5090 / static_compact_gpu_ridge: `{"model__alpha": 0.1}`: 18; `{"model__alpha": 0.01}`: 6
- RTX4090_to_RTX5090 / static_compact_gpu_trace_ridge: `{"model__alpha": 0.1}`: 15; `{"model__alpha": 1.0}`: 5; `{"model__alpha": 10.0}`: 3; `{"model__alpha": 0.01}`: 1
- RTX4090_to_RTX5090 / static_hpct_proxy_ridge: `{"model__alpha": 0.01}`: 19; `{"model__alpha": 1.0}`: 3; `{"model__alpha": 0.1}`: 2
- RTX4090_to_RTX5090 / static_only_ridge: `{"model__alpha": 0.01}`: 15; `{"model__alpha": 1.0}`: 7; `{"model__alpha": 0.1}`: 2
- RTX4090_to_RTX5090 / static_trace_hpct_proxy_ridge: `{"model__alpha": 1.0}`: 24
- RTX4090_to_RTX5090 / static_trace_ridge: `{"model__alpha": 1.0}`: 24
- RTX4090_to_RTX5090 / trace_only_ridge: `{"model__alpha": 1.0}`: 21; `{"model__alpha": 0.1}`: 3
- RTX5090_to_RTX4090 / full_profile_fusion_ridge: `{"model__alpha": 10.0}`: 18; `{"model__alpha": 1.0}`: 6
- RTX5090_to_RTX4090 / static_compact_gpu_hpct_proxy_ridge: `{"model__alpha": 0.1}`: 21; `{"model__alpha": 1.0}`: 3
- RTX5090_to_RTX4090 / static_compact_gpu_ridge: `{"model__alpha": 0.1}`: 21; `{"model__alpha": 0.01}`: 3
- RTX5090_to_RTX4090 / static_compact_gpu_trace_ridge: `{"model__alpha": 10.0}`: 20; `{"model__alpha": 1.0}`: 4
- RTX5090_to_RTX4090 / static_hpct_proxy_ridge: `{"model__alpha": 10.0}`: 17; `{"model__alpha": 1.0}`: 6; `{"model__alpha": 0.1}`: 1
- RTX5090_to_RTX4090 / static_only_ridge: `{"model__alpha": 1.0}`: 13; `{"model__alpha": 10.0}`: 10; `{"model__alpha": 0.1}`: 1
- RTX5090_to_RTX4090 / static_trace_hpct_proxy_ridge: `{"model__alpha": 1.0}`: 23; `{"model__alpha": 0.01}`: 1
- RTX5090_to_RTX4090 / static_trace_ridge: `{"model__alpha": 1.0}`: 23; `{"model__alpha": 0.01}`: 1
- RTX5090_to_RTX4090 / trace_only_ridge: `{"model__alpha": 1.0}`: 23; `{"model__alpha": 0.01}`: 1

## Worst workloads for the best Part 4 model

### RTX4090_to_RTX5090: `static_compact_gpu_ridge`

| Workload | Actual ms | Predicted ms | APE |
|---|---:|---:|---:|
| C01 | 29.060 | 25.940 | 10.74% |
| C16 | 41.930 | 39.087 | 6.78% |
| C13 | 30.371 | 32.367 | 6.57% |
| C02 | 28.075 | 29.809 | 6.18% |
| C09 | 30.749 | 32.236 | 4.84% |

### RTX5090_to_RTX4090: `static_compact_gpu_ridge`

| Workload | Actual ms | Predicted ms | APE |
|---|---:|---:|---:|
| C10 | 49.200 | 44.170 | 10.22% |
| C17 | 38.002 | 34.560 | 9.06% |
| C09 | 31.867 | 34.640 | 8.70% |
| C06 | 34.708 | 37.431 | 7.85% |
| C21 | 35.460 | 37.809 | 6.62% |


## 解讀檢查清單

- 區分 OOF evaluation 與重新用全部資料訓練的 production bundle。
- 不把 target runtime、oracle marker 或 future information 當作 deployment feature。
- 報告方向別結果、combined summary 與 worst cases。
- 結論只適用於目前資料、GPU 節點與 workload family。